# Requête HTTP 

Un requête HTTP est une requête basée sur le protocole TCP, elle fait partie de la couche application de la couche OSI. Elle permet d'accéder aux données mise à disposition sur une adresse IP (ou url résolue par un DNS) et un port. 

Les deux ports les plus utilisés dans le web sont le 80 pour les sites en HTTP et le 443 pour les sites en HTTPS. HTTPS est une variable du protocole HTTP basé sur le protocole TLS.

Il existe de nombreux types de requêtes selon la convention `REST`: 
- GET
- POST
- PUT 
- DELETE
- UPDATE.

Dans notre cas, nous allons utiliser la plupart du temps des GET et potentiellement des POST. 
- Le GET permet comme son nom l'indique de récupérer des informations en fonction de certains paramètres. 
- Le POST nécessite un envoi de données pour récupérer des données. Le body du post est, la plupart du temps, envoyé sous la forme d'un objet JSON.

Ces requêtes encapsulent un certain nombre de paramètres qui permettent soient d'identifier une provenance et un utilisateur ou de réaliser différentes actions.

In [1]:
import requests

In [2]:
url = "https://www.esiee.fr/"
response = requests.get(url, timeout=10)
response.status_code

200

Il existe deux méthodes pour récupérer le contenu de la page :

- `response.text` qui permet de retourner le texte sous la forme d'une chaine de charactères.
- `response.content` qui permet de récupérer le contenu de la page sous la forme de bytes

In [3]:
type(response.content)

bytes

In [4]:
type(response.text)

str

Pour récupérer les 1000 premiers charactères de la page :

In [5]:
response.text[0:1000]

'<!DOCTYPE html>\n<html lang="fr-FR">\n<head>\n\n<meta charset="utf-8">\n<!-- \n\tThis website is powered by TYPO3 - inspiring people to share!\n\tTYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.\n\tTYPO3 is copyright 1998-2026 of Kasper Skaarhoj. Extensions are copyright of their respective owners.\n\tInformation and contribution at https://typo3.org/\n-->\n\n\n\n<title>ESIEE Paris, l&#039;école de l&#039;innovation technologique | ESIEE Paris</title>\n<meta name="generator" content="TYPO3 CMS" />\n<meta name="description" content="Rejoignez ESIEE Paris, grande école d&#039;ingénieur dans les domaines des transitions numérique, énergétique et environnementale. Classée parmi les meilleures écoles d&#039;ingénieur selon le classement de l&#039;Etudiant. Habilitée par la Commission des Titres d&#039;Ingénieur (CTI). Membre de la Conférence des Grandes Ecoles (CGE). " />\n<meta name="viewport" content="width=device-w

Pour récupérer les headers HTTP de la réponse :

In [6]:
response.headers

{'Date': 'Tue, 17 Nov 2026 19:00:22 GMT', 'Server': 'Apache', 'Content-Language': 'fr', 'Vary': 'Accept-Encoding', 'Content-Encoding': 'gzip', 'X-UA-Compatible': 'IE=edge', 'X-Content-Type-Options': 'nosniff', 'Content-Length': '17417', 'Content-Type': 'text/html; charset=utf-8', 'X-Varnish': '55739699 55411052', 'Age': '65', 'Via': '1.1 varnish (Varnish/7.1)', 'Accept-Ranges': 'bytes', 'Connection': 'keep-alive'}

On peut modifier les paramètres de la requête et/ou ses headers. On peut par exemple ajouter un UserAgent (identifiant de l'initiateur de la requête) et un timeout de 10 secondes :

In [7]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
response = requests.get(url, headers=headers, timeout = 10)
response.content[0:1000]

b'<!DOCTYPE html>\n<html lang="fr-FR">\n<head>\n\n<meta charset="utf-8">\n<!-- \n\tThis website is powered by TYPO3 - inspiring people to share!\n\tTYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.\n\tTYPO3 is copyright 1998-2026 of Kasper Skaarhoj. Extensions are copyright of their respective owners.\n\tInformation and contribution at https://typo3.org/\n-->\n\n\n\n<title>ESIEE Paris, l&#039;\xc3\xa9cole de l&#039;innovation technologique | ESIEE Paris</title>\n<meta name="generator" content="TYPO3 CMS" />\n<meta name="description" content="Rejoignez ESIEE Paris, grande \xc3\xa9cole d&#039;ing\xc3\xa9nieur dans les domaines des transitions num\xc3\xa9rique, \xc3\xa9nerg\xc3\xa9tique et environnementale. Class\xc3\xa9e parmi les meilleures \xc3\xa9coles d&#039;ing\xc3\xa9nieur selon le classement de l&#039;Etudiant. Habilit\xc3\xa9e par la Commission des Titres d&#039;Ing\xc3\xa9nieur (CTI). Membre de la Conf\xc3\x

## Exercice

## Exercice 1

- Créer une classe Python permettant de faire des requêtes HTTP.
- Cette classe doit utiliser toujours le même UserAgent.
- Le TimeOut sera spécifié à chaque appelle avec une valeur par défaut.
- Un mécanisme de retry sera mis en place de façon recursive.

## Exercice 2

- Faire une fonction permettant de supprimer tous les espaces supperflus d'une string
- Faire une fonction qui prend une string html et renvois une string intelligible (enlever les caractères spéciaux,
- Récupérer le domaine en fonction d'un url

### Solution — exercice 1
Deux retries signifient trois tentatives au maximum.

In [8]:
import random
import time
from urllib.parse import urlparse, urljoin, parse_qs
import requests
from bs4 import BeautifulSoup

class HttpClient:
    """Client à User-Agent fixe ; retries récursifs bornés."""
    def __init__(self, user_agent="DataEngineerTools/1.0 (educational)"):
        self.user_agent = user_agent
        self.session = requests.Session()

    def get(self, url, timeout=10, retries=2, **kwargs):
        if not 0 <= retries <= 10:
            raise ValueError("retries doit être compris entre 0 et 10")
        headers = dict(kwargs.pop("headers", {}))
        headers["User-Agent"] = self.user_agent
        try:
            response = self.session.get(url, timeout=timeout, headers=headers, **kwargs)
            response.raise_for_status()
            return response
        except (requests.Timeout, requests.ConnectionError):
            if retries == 0:
                raise
        except requests.HTTPError as error:
            # Pas de nouvelle tentative sur une erreur client (403, 404...).
            if error.response.status_code not in (500, 502, 503, 504) or retries == 0:
                raise
        time.sleep(1)
        return self.get(url, timeout=timeout, retries=retries-1, headers=headers, **kwargs)

    def close(self):
        self.session.close()

### Solution — exercice 2
On conserve les accents et la ponctuation utiles.

In [9]:
def clean_spaces(text):
    """Réduit espaces, tabulations et retours à la ligne à un espace."""
    return " ".join(text.split())

def clean_html(html):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    # BeautifulSoup décode aussi les entités (&amp;, &nbsp;...).
    return clean_spaces(soup.get_text(" ", strip=True))

def get_domain(url):
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("Une URL absolue HTTP(S) est attendue")
    return parsed.hostname.lower()

assert clean_spaces("  Bonjour\n  Fami\t ! ") == "Bonjour Fami !"
assert clean_html("<p>Café &amp; thé</p><script>bad()</script>") == "Café & thé"
assert get_domain("https://example.org:443/article") == "example.org"

# Exploitation du HTML  

Ici, il faut récupérer le code HTML d'un site web à partir d'une requête. Lorsque vous avez récupéré le texte d'un site il faut le parser. Pour cela, on utilise BeautifulSoup qui permet de transformer la structure HTML en objet Python. Cela permet de récupérer efficacement les données qui nous intéresse.

Pour les webmasters, le blocage le plus souvent mis en place et un blocage sur le User-Agent. Le User-Agent est un paramètre intégré dans la requête HTTP réalisé par le Navigateur pour envoyer au front des informations basiques :

- la version du Navigateur,
- la version de l'OS
- Le type de gestionnaire graphique (Gecko)
- le type de device utilisé

Exemple de User Agent :  

`Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:47.0) Gecko/20100101 Firefox/47.0`

Commençons à utiliser `BeautifulSoup`, il est normalement déjà installé, le cas échéant executez les lignes suivantes : 

In [10]:
import requests
from bs4 import BeautifulSoup

Pour transformer une requête (requests) en objet BeautifulSoup :

In [11]:
response = requests.get(url, timeout=10)
soup = BeautifulSoup(response.text, "html.parser")

Pour trouver tous les liens d'une page, on récupère la balise `a` qui permet de gérer les liens en HTML :

In [12]:
soup.find_all("a")[0:10]

[<a href="/#content">Aller au contenu</a>,
 <a href="/#menu">Aller au menu</a>,
 <a href="/plan-du-site/">Plan du site</a>,
 <a href="/actualites/journees-portes-ouvertes-2026-2027" target="_blank" title="Ouvre une nouvelle fenêtre">Portes Ouvertes ESIEE Paris - Rendez-vous samedi 12 décembre de 13h à 18h </a>,
 <a href="/"><img alt="ESIEE PARIS" class="a42-ac-replace-img" src="/typo3conf/ext/esiee_sitepackage/Resources/Public/imgs/svg/logo-esiee.svg"/></a>,
 <a href="/brochures-1">Brochures</a>,
 <a href="/informations/etudiantes-et-etudiants">Espace élèves</a>,
 <a href="/" hreflang="fr-FR" title="Français">
 <span>Fr</span>
 </a>,
 <a href="/en/" hreflang="en-US" title="English">
 <span>En</span>
 </a>,
 <a href="/candidater-1">Candidater</a>]

On peut aussi préciser la classe HTML qu'on veut récupérer :

```python
soup.find_all(class_="<CLASS_NAME>")[0:10]
```

Ici par exemple: 

In [13]:
soup.find_all(class_="toggler")[0:5]

[<button aria-controls="searchbox-header-form" aria-expanded="false" class="toggler">
 <i class="fa-solid fa-magnifying-glass"></i>
 <i class="fa-solid fa-xmark"></i>
 <span class="sr-only">
 <span class="display">Afficher</span><span class="hide">Masquer</span> la recherche
 		</span>
 </button>,
 <button aria-controls="submenu-40" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>L'école</button>,
 <button aria-controls="submenu-563" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>Gouvernance et conseils</button>,
 <button aria-controls="submenu-65" aria-expanded="false" class="toggler"><span class="sr-only"><span class="display">Afficher</span><span class="hide">Masquer</span> le sous menu : </span>Départements d'enseignements et de recherche</button>,
 <button aria-controls="su

Pour récupérer le text sans les balises HTML :

In [14]:
soup.text[0:1000]

"\n\n\n\n\nESIEE Paris, l'école de l'innovation technologique | ESIEE Paris\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAller au contenu\nAller au menu\nPlan du site\n\n\n\n\n\n\n\nPortes Ouvertes ESIEE Paris - Rendez-vous samedi 12 décembre de 13h à 18h \n\n\n\n\n\nMasquer l'alerte\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nBrochuresEspace élèves\n\n\n\nFr\n\n\n\n\nEn\n\n\n\n\n\n\n\n\n\nAfficherMasquer la recherche\r\n\t\t\n\n\n\nSaisissez votre recherche\xa0:\n\nLancer la recherche\n\n\n\nCandidater\n\nAfficherMasquer le menu\n\n\n\n\n\nRetour au menu principalAfficherMasquer le sous menu\xa0: L'écolePourquoi choisir ESIEE Paris ?AfficherMasquer le sous menu\xa0: Gouvernance et conseilsGouvernance et conseilsConseil scientifiqueAfficherMasquer le sous menu\xa0: Départements d'enseignements et de rechercheInformatique et télécommunicationsIngénierie des systèmes cyberphysiquesIngénierie industrielleSanté, énergie et environnement durableManagement, sciences humaines et languesCorps professor

## Exercice
### Exercice 3

Améliorer la classe développé précédemment.

- Ajouter une méthode pour récupérer l'objet soup d'un url
- Récupérer une liste de User Agent et effectuer une rotation aléatoire sur celui à utiliser
- Utiliser cette classe pour parser une page HTML et récupérer : le titre, tous les H1 (si ils existent), les liens vers les images, les liens sortants vers d'autres sites, et le texte principal.

### Solution — exercice 3

In [15]:
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/130.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64; rv:132.0) Gecko/20100101 Firefox/132.0",
]

class Scraper(HttpClient):
    def get(self, url, **kwargs):
        # Rotation demandée par l'exercice 3 ; la classe de base reste fixe.
        self.user_agent = random.choice(USER_AGENTS)
        return super().get(url, **kwargs)

    def get_soup(self, url, **kwargs):
        response = self.get(url, **kwargs)
        return BeautifulSoup(response.content, "html.parser")

    def parse_page(self, html, url):
        soup = BeautifulSoup(html, "html.parser")
        def absolute(value):
            result = urljoin(url, value)
            return result if urlparse(result).scheme in ("http", "https") else None
        images = list(dict.fromkeys(filter(None, (absolute(img['src']) for img in soup.select('img[src]')))))
        links = list(dict.fromkeys(filter(None, (absolute(a['href']) for a in soup.select('a[href]')))))
        main = soup.find('article') or soup.find('main') or soup.body or soup
        # Heuristique : article, puis main ; sinon corps sans navigation.
        main = BeautifulSoup(str(main), "html.parser")
        for tag in main.select('nav, header, footer, aside, script, style, noscript'):
            tag.decompose()
        return {
            'url': url, 'domain': get_domain(url),
            'title': clean_spaces(soup.title.get_text()) if soup.title else '',
            'h1': [clean_spaces(h.get_text(' ', strip=True)) for h in soup.find_all('h1')],
            'images': images,
            'outgoing_links': [link for link in links if get_domain(link) != get_domain(url)],
            'text': clean_html(str(main)),
        }

    def scrape(self, url):
        response = self.get(url)
        return self.parse_page(response.content, response.url)

scraper = Scraper()
example_html = '<title>Exemple</title><main><h1>Bonjour</h1><p>Un article.</p><img src="/photo.jpg"><a href="https://other.org">Lien</a></main>'
scraper.parse_page(example_html, 'https://example.org/article')

{'url': 'https://example.org/article',
 'domain': 'example.org',
 'title': 'Exemple',
 'h1': ['Bonjour'],
 'images': ['https://example.org/photo.jpg'],
 'outgoing_links': ['https://other.org'],
 'text': 'Bonjour Un article. Lien'}

# Exploitation des appels d'API



Losque le front du site récupère des données sur une API gérée par le back, un appel d'API est réalisé. Cet appel est recensé dans les appels réseaux. Il est alors possible de re-jouer cet appel pour récupérer à nouveau les données. Il est très facile de récupérer ces appels dans l'onglet Network de la console développeur de Chrome ou FireFox. La console vous permet de copier le code CURL de la requête et vous pouvez ensuite la transformer en code Python depuis le site https://curl.trillworks.com/.

Souvent les APIs sont bloquées avec certains paramètres. L'API vérifie que dans les headers de la requête HTTP ces paramètres sont présents :
* un token généré à la volée avec des protocoles OAuth2 (ou moins développés).
* un referer provenant du site web (la source de la requête), très facile à falsifier.



## Exercice 
### Exercice 4

- Utiliser les informations développées plus haut pour récupérer les premiers résultats d'une recherche d'une requête
sur Google. 

Tips : 

- Ouvrir les outils de développements de Chrome ou Firefox
- Onglet Network
- Fouiller dans les requêtes pour voir à quoi ressemble un appel API Google
- Utilisez beautiful soup pour convertir le contenu de la request en objet et accéder aux balises

### Solution — exercice 4
Google renvoie du HTML, pas une API publique stable. Le parseur peut nécessiter une adaptation. Les résultats de démonstration ci-dessous sont fictifs.

In [16]:
def parse_google_results(html, base_url="https://www.google.com/search", limit=5):
    soup = BeautifulSoup(html, "html.parser")
    results, seen = [], set()
    for heading in soup.find_all('h3'):
        anchor = heading.find_parent('a', href=True)
        if not anchor:
            continue
        href = urljoin(base_url, anchor['href'])
        parsed = urlparse(href)
        if parsed.path == '/url':
            params = parse_qs(parsed.query)
            href = (params.get('q') or params.get('url') or [''])[0]
        parsed = urlparse(href)
        if parsed.scheme not in ('http', 'https') or not parsed.hostname:
            continue
        if parsed.hostname == 'google.com' or parsed.hostname.endswith('.google.com'):
            continue
        if href in seen:
            continue
        results.append({'title': clean_spaces(heading.get_text(' ', strip=True)), 'url': href})
        seen.add(href)
        if len(results) >= limit:
            break
    return results

def google_search(query, limit=5):
    if limit <= 0:
        return []
    response = scraper.get('https://www.google.com/search', params={'q': query, 'num': limit})
    results = parse_google_results(response.content, response.url, limit)
    if not results:
        raise RuntimeError("Aucun résultat exploitable : consentement, CAPTCHA ou HTML modifié. Ne pas contourner le blocage.")
    return results

# À lancer séparément pour une requête réelle :
# google_search('data engineering ESIEE')
parse_google_results('<a href="/url?q=https%3A%2F%2Fexample.org"><h3>Exemple</h3></a>')

[{'title': 'Exemple', 'url': 'https://example.org'}]

# Exercice Final  

Exercice Final
Utilisez tout ce que vous avez appris pour récupérer des articles de News avec une catégorie. Il est souvent intéressant de partir des flux RSS pour commencer :

Les données doivent comprendre :
- Le texte important propre
- L'url
- Le domaine
- la catégorie
- Le titre de l'article
- Le titre de la page
- (Facultatif) : les images

Tips : 

- Taper le nom de votre média favoris + RSS (par exemple : https://www.lemonde.fr/rss/)
- Aller dans le DOM de la page 
- Trouver les catégories et les liens vers les articles

### Solution — exercice final
La sélection du texte principal est heuristique ; vérifier le résultat pour chaque média. Les erreurs sont conservées séparément.

In [17]:
import json
from pathlib import Path
from xml.etree import ElementTree as ET

def collect_news(feed_url, category, limit=3, delay=1):
    """Lit un flux RSS 2.0 puis visite au maximum limit articles publics."""
    if limit <= 0:
        return {'articles': [], 'errors': []}
    response = scraper.get(feed_url)
    feed = ET.fromstring(response.content)
    articles, errors, seen = [], [], set()
    for item in feed.findall('./channel/item'):
        url = (item.findtext('link') or '').strip()
        if not url or url in seen:
            continue
        seen.add(url)
        if len(seen) > limit:
            break
        time.sleep(delay)
        try:
            data = scraper.scrape(url)
            if not data['text']:
                raise ValueError('Texte principal vide')
            articles.append({
                'text': data['text'], 'url': data['url'], 'domain': data['domain'],
                'category': item.findtext('category') or category,
                'article_title': data['h1'][0] if data['h1'] else clean_spaces(item.findtext('title') or ''),
                'page_title': data['title'], 'images': data['images'],
            })
        except (requests.RequestException, ValueError) as error:
            errors.append({'url': url, 'error': str(error)})
    return {'articles': articles, 'errors': errors}

# Exécution réseau volontairement séparée : vérifier les règles du média.
# Pas de contournement des pages réservées aux abonnés.
# news = collect_news('https://www.lemonde.fr/sciences/rss_full.xml', 'Sciences')
# Path('news.json').write_text(json.dumps(news, ensure_ascii=False, indent=2), encoding='utf-8')

### Exécution réelle du projet final
Le flux BBC contient des articles publics. Les résultats et les erreurs sont exportés séparément dans le même JSON.

In [18]:
news = collect_news('https://feeds.bbci.co.uk/news/science_and_environment/rss.xml', 'Science et environnement', limit=2)
Path('results').mkdir(exist_ok=True)
Path('results/news_bbc.json').write_text(json.dumps(news, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"Articles collectés : {len(news['articles'])} ; erreurs : {len(news['errors'])}")
for article in news['articles']:
    print(article['article_title'], '|', article['domain'])


Articles collectés : 2 ; erreurs : 0
William and Kate celebrate Earthshot Prize winners in Mumbai | www.bbc.co.uk
More than £160m needed to clean up one of Europe's largest illegal dumps | www.bbc.co.uk
